In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

# This notebook lives in backend/notebooks/; run from backend/ so the
# data/... paths below resolve.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)


# BDA 594 Group Project -- Bad Data Analytics
## Data Cleaning Notebook
### Written by Matthew Tu

---

##### This Notebook is meant to take disparate data from 3 different sources: San Francisco, Los Angelos, and San Diego.

##### In the best case scenario, we want to keep all rows where we can understand crime type, subcrime type, and any form of geo location as accurate as county if possible. 

##### We also want to be mindful about the amount of people that can be looped into the same incident. For this, we can count the number of times a case number appears and add it to a new column called "Number of Suspects" or/and "Number of Victims".

---

### First we have to do some EDA

In [12]:
# First, San Diego's datasets are separated by year from 2020 to 2026. We need to combine these datasets into one single CSV.
sd_combined = []
for year in range(2020, 2026 + 1):
    fp_SD = f'data/raw/sd/pd_nibrs_{year}_datasd.csv'
    pd_SD = pd.read_csv(fp_SD)
    sd_combined.append(pd_SD)

sd_combined = pd.concat(sd_combined, ignore_index=True)
sd_combined.to_csv("data/archive/pd_nibrs_2020_2026_combined.csv", index=False)


/var/folders/4f/6hcglkrd6tx6c1qzhyrfhfsh0000gn/T/ipykernel_56907/2916000891.py:5: DtypeWarning: Columns (2: case_number) have mixed types. Specify dtype option on import or set low_memory=False.
  pd_SD = pd.read_csv(fp_SD)


In [28]:
# 
    # Start with San Francisco
fp_SF = 'data/raw/sf/Police_Department_Incident_Reports__2018_to_Present_20260924.csv'
pd_SF = pd.read_csv(fp_SF)

n_rows_SF = pd_SF.shape[0]
n_unique_incidents_SF = pd_SF['Incident ID'].nunique()
#pd_SF.head()

statement = f""" 
Number of Rows in SF Data: {n_rows_SF}
Number of unique incident ID's in SF Data: {n_unique_incidents_SF}
Difference: {n_rows_SF - n_unique_incidents_SF} """
print(statement)

    # Now for Los Angeles
fp_LA = 'data/raw/la/LAPD_NIBRS_Victims_Dataset.csv'
pd_LA = pd.read_csv(fp_LA)
#pd_LA.head()

n_rows_LA = pd_LA.shape[0]
n_unique_incidents_LA = pd_LA['CaseNo'].nunique()

statement = f"""
Number of Rows in LA Data: {n_rows_LA}
Number of unique incident ID's in LA Data: {n_unique_incidents_LA}
Difference: {n_rows_LA - n_unique_incidents_LA}
"""
print(statement)

    # Lastly, San Diego
fp_SD = 'data/archive/pd_nibrs_2020_2026_combined.csv'
pd_SD = pd.read_csv(fp_SD)
#pd_SD.head()

n_rows_SD = pd_SD.shape[0]
n_unique_incidents_SD = pd_SD['case_number'].nunique()

statement = f"""
Number of Rows in SD Data: {n_rows_SD}
Number of unique incident ID's in SD Data: {n_unique_incidents_SD}
Difference: {n_rows_SD - n_unique_incidents_SD}
"""
print(statement)

 
Number of Rows in SF Data: 1067050
Number of unique incident ID's in SF Data: 883929
Difference: 183121 

Number of Rows in LA Data: 448973
Number of unique incident ID's in LA Data: 406675
Difference: 42298


Number of Rows in SD Data: 549409
Number of unique incident ID's in SD Data: 473831
Difference: 75578



/var/folders/4f/6hcglkrd6tx6c1qzhyrfhfsh0000gn/T/ipykernel_56907/3129017707.py:33: DtypeWarning: Columns (2: case_number) have mixed types. Specify dtype option on import or set low_memory=False.
  pd_SD = pd.read_csv(fp_SD)


> ### Findings

##### We find that San Francisco has 183121 cases with 2 or more suspects involved. Los Angeles had 41479. San Diego had 75578

| City | Years | Unique Crimes | Total Crime | Number of Multiple V/S crimes | 
| --- | --- | --- | --- | --- |
|San Francisco| 2018-2026|883929|1067050| 183121|
|Los Angeles|2024-2026|398289|439768|41479|
|San Diego|2020-2026|473831|549409|75578|

In [24]:
# list all columns in the San Diego dataset
pd_SD.columns.to_list()
# Preview Specific columns
pd_SD[['case_number',
       'occured_on',
       'approved_on',
       'code_section',
       'group_type',
       'ibr_category',
       'crime_against',
       'ibr_offense',
       'ibr_offense_description',
       'pd_offense_category',
       'beat',
       'neighborhood',
       'block_addr',
       'city',
       'zip',
       ]]

,case_number,occured_on,approved_on,code_section,group_type,ibr_category,crime_against,ibr_offense,ibr_offense_description,pd_offense_category,beat,neighborhood,block_addr,city,zip
0,20007076,2020-02-04,2020-02-05 14:23:11,BW-F ZZ FELONY BENCH WARRANT (OUR AGENCY) ||,B,"GROUP ""B"" OFFENSES",NaN,90Z,All Other Offenses,All Other Offenses,822.0,El Cerrito,5600 University AVE,SAN DIEGO,92105.0
1,20707004,2020-09-07,2020-09-30 16:09:12,488 PC PETTY THEFT(All Other Larceny) (M) ||,A,LARCENY/THEFT OFFENSES,PR,23H,All Other Larceny,Larceny,526.0,Cortez,NaN,NaN,92101.0
2,20704084,2020-05-27,2020-05-28 20:09:08,487 (A) PC GRAND THEFT (Theft From Mot Veh) (F...,A,LARCENY/THEFT OFFENSES,PR,23F,Theft From Motor Vehicle,Theft From Motor Vehicle,121.0,Mission Beach,3500 BAYSIDE WALK,SAN DIEGO,92109.0
3,20700486,2020-01-14,2020-01-16 10:34:38,487 (A) PC GRAND THEFT (Theft From Mot Veh) (F...,A,LARCENY/THEFT OFFENSES,PR,23F,Theft From Motor Vehicle,Theft From Motor Vehicle,523.0,Gaslamp,500 K STREET,SAN DIEGO,92101.0
4,20009409,2020-02-14,2020-02-15 01:14:55,594 (B)(1) PC VANDALISM ($400 OR MORE) (F) ||,A,DESTRUCTION/DAMAGE/VANDALISM OF PROPERTY,PR,290,Destruction/Damage/Vandalism of Property,Vandalism,826.0,Colina Del Sol,NaN,SAN DIEGO,92105.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
549404,26718289,2026-09-21,2026-09-23 21:31:18,488 PC PETTY THEFT(Shoplift) (M) ||,A,LARCENY/THEFT OFFENSES,PR,23C,Shoplifting,Larceny,723.0,Otay Mesa West,2900 CORONADO (SB) AVE,SD,92154.0
549405,26302950,2026-09-14,2026-09-23 23:37:55,10851 (A) VC TAKE VEHICLE W/O OWNER'S CONSENT/...,A,MOTOR VEHICLE THEFT,PR,240,Motor Vehicle Theft,Motor Vehicle Theft,832.0,Teralta West,32nd ST & orange AVE,SD,92105.0
549406,26035708,2026-09-23,2026-09-23 23:47:15,1203.2 (A) PC PROBATION VIOLATION:REARREST/REV...,B,"GROUP ""B"" OFFENSES",NaN,90Z,All Other Offenses,All Other Offenses,433.0,Encanto,6200 Akins AVE,SD,92114.0
549407,26718291,2026-09-23,2026-09-23 21:41:28,530.5 (A) PC GET CREDIT/ETC OTHER'S ID (F) ||,A,FRAUD OFFENSES,PR,26C,Impersonation,Fraud/Counterfeiting,312.0,Tierra Santa,11500 CAMINO PLAYA CATALINA,SD,92124.0


### Listing all available columns for comparison. 


In [29]:
sf_col = pd_SF.columns.to_list()
la_col = pd_LA.columns.to_list()
sd_col = pd_SD.columns.to_list()

print("sf columns", sf_col)
print("la columns", la_col)
print("sd columns", sd_col)

sf columns ['Row ID', 'Incident Date', 'Incident Time', 'Incident Year', 'Incident Day of Week', 'Report Datetime', 'Incident ID', 'Incident Number', 'Filed Online', 'Incident Code', 'Incident Category', 'Incident Subcategory', 'Incident Description', 'Resolution', 'Intersection', 'Analysis Neighborhood', 'Latitude', 'Longitude', 'Point']
la columns ['CaseNo', 'UniqueVictimNo', 'Date Rptd', 'Date OCC', 'Time OCC', 'AREA NAME', 'TotalVictimCount', 'VictimOfsList', 'Vict Age', 'Status', 'Status Desc', 'Victim Shot']
sd columns ['objectid', 'nibrs_uniq', 'case_number', 'occured_on', 'approved_on', 'day_of_week', 'month', 'year', 'code_section', 'group_type', 'ibr_category', 'crime_against', 'ibr_offense', 'ibr_offense_description', 'pd_offense_category', 'violent_crime', 'property_crime', 'beat', 'neighborhood', 'service_area', 'division_number', 'division', 'block_addr', 'city', 'state', 'zip', 'geocode_status', 'geocode_score', 'latitude', 'longitude']


#### Findings
##### San Francisco Data
- Columns to drop: ['Row ID'].
##### Los Angeles Data
- No columns need to be dropped.
##### San Diego Data
- Columns to drop: ['objectid', 'approved_on', 'code_section', 'group_type', 'crime_against', 'ibr_offense', 'beat', ]

##### Potentially Related Fields

These are candidate relationships based on column names, not confirmed equivalents.

| Concept | San Francisco | Los Angeles | San Diego |
| --- | --- | --- | --- |
| Incident or case identifier | Incident ID / Incident Number | CaseNo | case_number |
| Occurrence date | Incident Date | Date OCC | occured_on |
| Occurrence time | Incident Time | Time OCC | Inspect occured_on |
| Reporting or processing date | Report Datetime | Date Rptd | approved_on — verify meaning |
| Offense information | Incident Code / Category / Subcategory / Description | VictimOfsList — inspect contents | ibr_category / ibr_offense / ibr_offense_description / pd_offense_category |
| Geographic area | Analysis Neighborhood | AREA NAME | neighborhood / service_area / division |
| Coordinates | Latitude / Longitude | Not present in supplied columns | latitude / longitude |

The geographic fields may describe areas of very different sizes, so they should not automatically be treated as equivalent.

##### Proposed Approach
1. Preserve the raw datasets and work on copies.
2. Confirm what one row represents in each dataset. Counting rows could otherwise compare victims in one city with offenses or incidents in another.
3. Inspect identifier uniqueness, missing values, date formats, and offense categories.
4. Choose a shared date range and offense scope.
5. Standardize values and data types while retaining the original column names.
6. Select columns for the final analysis after deciding which comparisons you will make.

### We will save the cleaned up datasets to CleanedData/...



In [47]:
# Select related columns while preserving their original names.
sf_desired_columns = [
    "Incident ID",
    "Incident Number",
    "Incident Date",
    "Incident Time",
    "Report Datetime",
    "Incident Code",
    "Incident Category",
    "Incident Subcategory",
    "Incident Description",
    "Analysis Neighborhood",
    "Latitude",
    "Longitude",
]

sf_subset = pd_SF[sf_desired_columns].copy()

# Save a new CSV in the current working directory.
from pathlib import Path

output_fp = Path("data/archive/CleanedData/SF/sf_clean.csv")
sf_subset.to_csv(output_fp, index=False)

# Los Angeles: select related columns while preserving original names.
from pathlib import Path

la_desired_columns = [
    "CaseNo",
    "Date OCC",
    "Time OCC",
    "Date Rptd",
    "VictimOfsList", 
    "UniqueVictimNo",
    "AREA NAME",
]

la_subset = pd_LA[la_desired_columns].copy()

la_output_fp = Path("data/archive/CleanedData/LA/la_clean.csv")
la_output_fp.parent.mkdir(parents=True, exist_ok=True)
la_subset.to_csv(la_output_fp, index=False)

# San Diego: select related columns while preserving original names.
from pathlib import Path

sd_desired_columns = [
    "case_number",
    "occured_on",
    "approved_on",
    "ibr_category",
    "ibr_offense",
    "ibr_offense_description",
    "pd_offense_category",
    "neighborhood",
    "latitude",
    "longitude",
]

sd_subset = pd_SD[sd_desired_columns].copy()

sd_output_fp = Path("data/archive/CleanedData/SD/sd_clean.csv")
sd_output_fp.parent.mkdir(parents=True, exist_ok=True)
sd_subset.to_csv(sd_output_fp, index=False)

That row groups several different kinds of offense information together. These columns are candidates for comparison, but they are not confirmed equivalents.

| Offense detail | San Francisco | Los Angeles | San Diego |
| --- | --- | --- | --- |
| Offense code | Incident Code | VictimOfsList — inspect contents | ibr_offense |
| Offense category | Incident Category | No separate category column supplied | ibr_category / pd_offense_category |
| Offense subcategory | Incident Subcategory | No separate subcategory column supplied | No confirmed equivalent |
| Offense description | Incident Description | VictimOfsList — inspect contents | ibr_offense_description |

The biggest uncertainty is `VictimOfsList`: its name suggests a list, but we need to inspect actual values to determine whether it contains codes, descriptions, or multiple offenses.

We also need to inspect SF and SD values before treating their codes or categories as equivalent. Similar column names do not establish a shared classification system.

In [48]:
# Inspect distinct offense values in each city.
offense_columns = {
    "SF": [
        "Incident Code",
        "Incident Category",
        "Incident Subcategory",
        "Incident Description",
    ],
    "LA": ["VictimOfsList"],
    "SD": [
        "ibr_category",
        "ibr_offense",
        "ibr_offense_description",
        "pd_offense_category",
    ],
}

city_data = {"SF": pd_SF, "LA": pd_LA, "SD": pd_SD}

with pd.option_context("display.max_colwidth", None):
    for city, columns in offense_columns.items():
        print(f"\n{city}: sample offense combinations")
        display(city_data[city][columns].drop_duplicates().head(20))


SF: sample offense combinations


,Incident Code,Incident Category,Incident Subcategory,Incident Description
0,51040,Non-Criminal,Non-Criminal,Aided Case
1,4134,Assault,Simple Assault,Battery
2,7021,Motor Vehicle Theft,Motor Vehicle Theft,"Vehicle, Stolen, Auto"
3,7041,Recovered Vehicle,Recovered Vehicle,"Vehicle, Recovered, Auto"
6,7043,Recovered Vehicle,Recovered Vehicle,"Vehicle, Recovered, Motorcycle"
7,9215,Forgery And Counterfeiting,Forgery And Counterfeiting,"Counterfeit Coins Or Notes, Possessing or Receiving"
8,72000,Non-Criminal,Non-Criminal,Found Property
10,27175,Other Miscellaneous,Other,Evading a Police Officer Recklessly
11,27170,Other Miscellaneous,Other,"Resisting, Delaying, or Obstructing Peace Officer Duties"
12,4074,Assault,Aggravated Assault,"Assault, Aggravated, On Police Officer, W/ Force"



LA: sample offense combinations


,VictimOfsList
0,812 - CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 YRS OLDER)
1,230B - ATTEMPT MURDER/HOMICIDE
2,110 - CRIMINAL HOMICIDE
3,230A - ASSAULT WITH DEADLY WEAPON
4,210B - ROBBERY - BUSINESS
5,"113 - MANSLAUGHTER, NEGLIGENT"
6,290.012(A) - PC - F - Sex Offender Fail Update Annual Registration - 90Z
7,845 - SEX OFFENDER REGISTRANT OUT OF COMPLIANCE
19,530.5(A) - PC - F - Identity Theft - 26F
21,290.015 - PC - M - 290 Violation After Incarceration - 90Z



SD: sample offense combinations


,ibr_category,ibr_offense,ibr_offense_description,pd_offense_category
0,"GROUP ""B"" OFFENSES",90Z,All Other Offenses,All Other Offenses
1,LARCENY/THEFT OFFENSES,23H,All Other Larceny,Larceny
2,LARCENY/THEFT OFFENSES,23F,Theft From Motor Vehicle,Theft From Motor Vehicle
4,DESTRUCTION/DAMAGE/VANDALISM OF PROPERTY,290,Destruction/Damage/Vandalism of Property,Vandalism
5,LARCENY/THEFT OFFENSES,23C,Shoplifting,Larceny
11,LARCENY/THEFT OFFENSES,23G,Theft of Motor Vehicle Parts/Accessories,Larceny
14,DRUG/NARCOTIC OFFENSES,35B,Drug Equipment Violations,Drug/Narcotic Violations
15,FRAUD OFFENSES,26C,Impersonation,Fraud/Counterfeiting
17,ARSON,200,Arson,Arson
25,MOTOR VEHICLE THEFT,240,Motor Vehicle Theft,Motor Vehicle Theft


`la_clean.csv`contains **448,973 rows** and **7,262 distinct `VictimOfsList` values**.

Spaced hyphens separate fields, but they also occur inside offense descriptions. There are multiple formats, so splitting every value into the same fixed number of columns would misalign information.

For example:

| Actual value | Interpretation |
| --- | --- |
| `210B - ROBBERY - BUSINESS` | Local offense code plus description containing a hyphen |
| `243(A) - PC - M - Battery On Person - Simple - 13B` | Statute, legal code, offense classification, description, and NIBRS code |
| `Shoplifting` | Description only |

In the structured statute format, `F` and `M` appear to indicate **felony and misdemeanor**, not gender.

Some cells also contain **multiple offenses**, such as:

`459 - PC - F - Burglary - Residential - 220, 594(B)(1) - PC - F - Vandalism ($400 Or More) - Felony - 290`

A simple comma split would also be unsafe because descriptions can contain commas—for example, `MANSLAUGHTER, NEGLIGENT`.

#### Proposed structure

Preserve `VictimOfsList` and add extracted columns. For a structured entry such as `243(A) - PC - M - Battery On Person - Simple - 13B`:

| New column | Example value |
| --- | --- |
| Statute | 243(A) |
| Legal Code | PC |
| Offense Classification | M |
| Local Offense Code | Missing for this format |
| Offense Description | Battery On Person - Simple |
| NIBRS Code | 13B |
| Crime Category | Assault Offenses |
| Crime Subcategory | Simple Assault |

The category and subcategory should come from a verified code lookup, not from splitting the description. Preserve the complete description so details such as residential burglary or business robbery remain available.

#### How to align the cities

1. **Separate multiple offenses carefully.** Create one row per source record–offense combination, retaining the source record identifier and `CaseNo`.
2. **Parse each format separately.** For structured statute entries, extract the first three fields and the final NIBRS code; preserve the text between them as the description. Handle local-code entries separately.
3. **Flag unmatched values for review.** Keep description-only or unusual entries without inventing missing codes.
4. **Map offenses into shared categories and subcategories.** NIBRS is a useful starting point for LA and SD. SF will need a reviewed mapping from its own codes and descriptions.
5. **Retain local detail alongside shared categories.** Matching the cities does not require discarding their more specific descriptions.

#### Column Splitting Approach

Use two steps: separate recognizable offense entries, then extract their fields. The actual column name is `VictimOfsList` (without an “s” after “Victim”).

The pattern below preserves hyphens inside descriptions and splits commas only when another recognizable offense follows. On your file, it produced 480,168 offense entries: 479,075 matched the statute format, 1,020 matched the local-code format, and 73 required review. These are pattern matches, not confirmation that every code is valid.

The original column stays intact. Expanded rows represent source record–offense combinations, not unique incidents.

In [51]:
import pandas as pd
from pathlib import Path

# Rebuild from the source every run so the cell can be safely re-run.
la_offenses = la_subset.copy()

# Split only before a recognizable statute entry or local offense code.
offense_separator = (
    r",\s*(?=(?:"
    r"[^,]+?\s+-\s+[A-Z]{2,5}\s+-\s+[FMI]\s+-\s+"
    r"|\d{3}[A-Z]?\s+-\s+"
    r"))"
)

la_offenses["Offense_Entry"] = (
    la_offenses["VictimOfsList"]
    .astype("string")
    .str.split(offense_separator, regex=True)
)

# One row per offense (a victim can have several).
la_offenses = la_offenses.explode("Offense_Entry", ignore_index=True)
la_offenses["Offense_Entry"] = la_offenses["Offense_Entry"].str.strip()

# Format: statute - legal code - classification - description - NIBRS code
statute_pattern = (
    r"^(?P<Statute>.+?)\s+-\s+"
    r"(?P<Legal_Code>[A-Z]{2,5})\s+-\s+"
    r"(?P<Offense_Classification>[FMI])\s+-\s+"
    r"(?P<Offense_Description>.+)\s+-\s+"
    r"(?P<NIBRS_Code>[0-9]{2}[0-9A-Z])$"
)

# Format: local offense code - full description
local_pattern = (
    r"^(?P<Local_Offense_Code>\d{3}[A-Z]?)\s+-\s+"
    r"(?P<Local_Description>.+)$"
)

statute_fields = la_offenses["Offense_Entry"].str.extract(statute_pattern)

local_fields = (
    la_offenses["Offense_Entry"]
    .where(statute_fields["Statute"].isna())
    .str.extract(local_pattern)
)

la_offenses = la_offenses.join(statute_fields)
la_offenses["Local_Offense_Code"] = local_fields["Local_Offense_Code"]

la_offenses["Offense_Description"] = (
    la_offenses["Offense_Description"]
    .fillna(local_fields["Local_Description"])
    .str.strip()
)

la_offenses["Parse_Status"] = "Needs review"
la_offenses.loc[la_offenses["Statute"].notna(), "Parse_Status"] = "Statute format"
la_offenses.loc[la_offenses["Local_Offense_Code"].notna(), "Parse_Status"] = "Local code format"

display(la_offenses.head())

# Inspect entries that did not match either format.
print(la_offenses["Parse_Status"].value_counts())
print(f"Victims: {la_offenses['UniqueVictimNo'].nunique():,} | Offense rows: {len(la_offenses):,}")

review_entries = (
    la_offenses.loc[la_offenses["Parse_Status"].eq("Needs review"), ["Offense_Entry"]]
    .value_counts()
    .reset_index(name="Count")
)

with pd.option_context("display.max_colwidth", None):
    display(review_entries)

# Save separately so the existing cleaned file is preserved.
output_fp = Path("data/archive/CleanedData/LA/la_offenses.csv")
output_fp.parent.mkdir(parents=True, exist_ok=True)
la_offenses.to_csv(output_fp, index=False)
print(f"Saved {len(la_offenses):,} rows to {output_fp}")


,CaseNo,Date OCC,Time OCC,Date Rptd,VictimOfsList,UniqueVictimNo,AREA NAME,Offense_Entry,Statute,Legal_Code,Offense_Classification,Offense_Description,NIBRS_Code,Local_Offense_Code,Parse_Status
0,201515681,2006 Oct 01 12:00:00 AM,2000,2020 Sep 29 12:00:00 AM,812 - CRM AGNST CHLD (13 OR UNDER) (14-15 & S...,201515681_0,N Hollywood,812 - CRM AGNST CHLD (13 OR UNDER) (14-15 & S...,NaN,NaN,NaN,CRM AGNST CHLD (13 OR UNDER) (14-15 & SUSP 10 ...,NaN,812,Local code format
1,201806745,2020 Feb 17 12:00:00 AM,2314,2020 Feb 18 12:00:00 AM,230B - ATTEMPT MURDER/HOMICIDE,201806745_0,Southeast,230B - ATTEMPT MURDER/HOMICIDE,NaN,NaN,NaN,ATTEMPT MURDER/HOMICIDE,NaN,230B,Local code format
2,201818467,2020 Oct 10 12:00:00 AM,1000,2020 Oct 12 12:00:00 AM,110 - CRIMINAL HOMICIDE,201818467_0,Southeast,110 - CRIMINAL HOMICIDE,NaN,NaN,NaN,CRIMINAL HOMICIDE,NaN,110,Local code format
3,201820975,2020 Nov 22 12:00:00 AM,1730,2020 Nov 22 12:00:00 AM,230A - ASSAULT WITH DEADLY WEAPON,201820975_0,Southeast,230A - ASSAULT WITH DEADLY WEAPON,NaN,NaN,NaN,ASSAULT WITH DEADLY WEAPON,NaN,230A,Local code format
4,201900963,2020 Dec 11 12:00:00 AM,1445,2020 Dec 13 12:00:00 AM,210B - ROBBERY - BUSINESS,201900963_0,Mission,210B - ROBBERY - BUSINESS,NaN,NaN,NaN,ROBBERY - BUSINESS,NaN,210B,Local code format


Parse_Status
Statute format       479075
Local code format      1020
Needs review             73
Name: count, dtype: int64
Victims: 448,973 | Offense rows: 480,168


,Offense_Entry,Count
0,14601.2(A) - VC - M - Driv - Susp/Etc Lic - Dui - Vio -,20
1,Grand Theft - Money/Labor/Property,9
2,Burglary,7
3,21453(A) - VC - I - FAIL STOP LINE/ETC AT RED -,7
4,16028(A) - VC - I - FAIL PROVE FIN RSP:PO REQ -,4
5,Vandalism ($400 Or More),3
6,battery,2
7,243(A) - PC - M - Battery On Person - Simple - 40,2
8,THEFT IDENTITY,1
9,314.1,1


Saved 480,168 rows to BDA594_Project_Data/CleanedData/LA/la_offenses.csv


In [2]:
import sys
!"{sys.executable}" "data/archive/CleanedData/LA/check_la_offenses.py"


Source: 448,973 victims | Parsed: 480,168 offense rows

Completeness
[PASS] Every source victim appears in the parsed file - 0 missing
[PASS] No victims appear that are not in the source - 0 extra
[PASS] Parsed file has at least one row per victim - 480,168 rows vs 448,973 victims
[PASS] No blank offense entries - 0 blank
[PASS] Split entries rejoin to the original VictimOfsList - 0 victims differ
[PASS] Source columns unchanged after explode - 0 rows differ

Parse coverage
       Statute format       479,075  (99.772%)
       Local code format      1,020  (0.212%)
       Needs review              73  (0.015%)
[PASS] Parse rate at least 99.9% - 99.985%
[PASS] Parse_Status only has expected values - unexpected: set()

Field consistency
[PASS] Statute rows have every statute field - 0 incomplete
[PASS] Local rows have a code and description - 0 incomplete
[PASS] No row is both statute and local format - 0 rows
[PASS] 'Needs review' rows have no parsed fields - 0 rows
[PASS] Classificatio

## Packaged pipeline

The cleaning above is now the `crime_pipeline` package (`backend/pipeline/crime_pipeline/`). It rebuilds `data/cleaned/`, `data/standardized/` and `data/crime.db` from the raw files. See `README.md` and `DATA_CONTRACT.md`.


In [ ]:
import sys

# clean -> standardize -> check -> load (about 2.5 minutes)
!"{sys.executable}" -m crime_pipeline all